In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DATA_DIR        = Path("../data")
REPORT_DIR      = DATA_DIR / "reports"
DRIFT_THRESHOLD = 0.2  # alert if >20% of features drift

REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

reference = pd.read_csv(DATA_DIR / "X_train.csv", index_col=0)
current   = pd.read_csv(DATA_DIR / "X_test.csv",  index_col=0)

print(f"Reference (train) : {len(reference)} rows")
print(f"Current   (test)  : {len(current)} rows")
print(f"Features          : {list(reference.columns)}")

In [ ]:
from evidently import ColumnMapping
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.report import Report

report = Report(metrics=[DataDriftPreset(), DataQualityPreset()])
report.run(
    reference_data=reference,
    current_data=current,
    column_mapping=ColumnMapping(),
)

report_path = REPORT_DIR / "drift_report.html"
report.save_html(str(report_path))
print(f"Report saved to {report_path}")

In [ ]:
result       = report.as_dict()
drift_result = result["metrics"][0]["result"]

share_drifted = drift_result["share_of_drifted_columns"]
n_drifted     = drift_result["number_of_drifted_columns"]
n_total       = len(reference.columns)

print(f"Drifted columns: {n_drifted}/{n_total} ({share_drifted:.1%})")

if share_drifted > DRIFT_THRESHOLD:
    print(f"ALERT: drift {share_drifted:.1%} exceeds threshold {DRIFT_THRESHOLD:.1%}")
    print("Consider triggering a retraining pipeline.")
else:
    print("No significant drift detected.")

In [ ]:
import joblib
from sklearn.metrics import accuracy_score, roc_auc_score

model  = joblib.load(DATA_DIR / "model.pkl")
y_test = pd.read_csv(DATA_DIR / "y_test.csv", index_col=0).squeeze()

y_pred = model.predict(current)
y_prob = model.predict_proba(current)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Performance on current data:")
print(f"  Accuracy : {acc:.4f}")
print(f"  ROC AUC  : {auc:.4f}")